# WristVoice — train a tiny on-device ASR by distilling from Whisper

**What this notebook does (end to end, on a free Colab GPU):**
1. Pull a real speech dataset (LibriSpeech dev-clean).
2. Use **Whisper** (the teacher) to label the audio — the distillation data.
3. Train **your small streaming student** (~11 M params) on those labels.
4. Measure WER, then **export + int4-quantize** to a ~5–6 MB model.

> **First:** `Runtime → Change runtime type → GPU (T4)`.

> **Honest scope:** this is a *scaled demo* on a small subset so it finishes in ~1 hour. 
> A from-scratch student on this little data won't be production-accurate — accuracy scales 
> with data + steps (run `train-clean-100`/`360` on your office GPU for real numbers). 
> What it *does* prove: the real pipeline (teacher → labels → student → quantized model) works on real speech.


## 1. Setup — clone the repo, check the GPU, install deps
We keep Colab's CUDA build of `torch`/`torchaudio` (don't reinstall them) and only add the extras. 
`torchaudio` being present switches on the **fast CUDA RNN-T loss** automatically.


In [ ]:
!nvidia-smi -L
import os
%cd /content
if os.path.exists('/content/ASR/.git'):
    !cd /content/ASR && git pull -q     # get the latest fixes on rerun
else:
    !git clone -q https://github.com/codejawk/ASR.git
%cd /content/ASR
import importlib, subprocess, sys
def ensure(pkg, pipname=None):
    try: importlib.import_module(pkg)
    except ImportError: subprocess.run([sys.executable,'-m','pip','install','-q',pipname or pkg])
# extras only — DO NOT reinstall torch/torchaudio on Colab
for p,q in [('faster_whisper','faster-whisper'),('soundfile','soundfile'),
            ('sentencepiece','sentencepiece'),('onnx','onnx'),('onnxruntime','onnxruntime'),('onnxscript','onnxscript')]:
    ensure(p,q)
import torch, torchaudio
print('torch',torch.__version__,'| torchaudio',torchaudio.__version__,'| CUDA',torch.cuda.is_available())
import edge_asr, edge_asr.losses.rnnt as R; print('fast RNN-T kernel available:', R._HAS_TORCHAUDIO)


## 2. Get real speech — LibriSpeech dev-clean (CC-BY-4.0)
Downloads once (~340 MB). We take a subset so the demo is quick; bump `N_UTTS` for better accuracy. 
LibriSpeech ships ground-truth transcripts, which we keep **only to measure WER** — the student trains on Whisper's labels.


In [ ]:
import torchaudio, os, json, random, soundfile as sf
N_UTTS = 1200          # raise to 3000+ for better results (slower)
MAX_SEC = 16           # skip very long clips to avoid GPU OOM
os.makedirs('data/wav', exist_ok=True)
ds = torchaudio.datasets.LIBRISPEECH('data', url='dev-clean', download=True)
idx = list(range(len(ds))); random.Random(0).shuffle(idx)
rows = []
for i in idx:
    wav, sr, text, *_ = ds[i]
    assert sr == 16000
    dur = wav.shape[-1]/16000
    if dur > MAX_SEC: continue
    k = len(rows)
    p = os.path.abspath(f'data/wav/{k}.wav')
    sf.write(p, wav.squeeze().numpy(), 16000)
    rows.append({'audio': p, 'text': text.lower(), 'duration': round(dur,2)})
    if len(rows) >= N_UTTS: break
random.Random(1).shuffle(rows)
train_rows, eval_rows = rows[:-150], rows[-150:]
open('data/train_gt.jsonl','w').write('\n'.join(json.dumps(r) for r in train_rows))
open('data/eval.jsonl','w').write('\n'.join(json.dumps(r) for r in eval_rows))
open('data/train_text.txt','w').write('\n'.join(r['text'] for r in train_rows))
print('train', len(train_rows), '| eval', len(eval_rows), '| total hours ~', round(sum(r['duration'] for r in rows)/3600,2))


## 3. Whisper (teacher) labels the audio → distillation data
Whisper transcribes the training clips. We then check the teacher's own WER on the held-out set 
(this is the quality ceiling the student distills toward).


In [ ]:
# label every clip with Whisper (small = fast+decent; use 'medium'/'large-v3' for higher quality)
!python scripts/pseudo_label_whisper.py --audio-dir data/wav --out data/pseudo.jsonl \
    --model small --device cuda --compute-type float16

import json, os
pseudo = {os.path.abspath(json.loads(l)['audio']): json.loads(l)['text'].lower() for l in open('data/pseudo.jsonl')}
# build the STUDENT's training file = Whisper labels for the train split
with open('data/train_pseudo.jsonl','w') as f:
    for r in [json.loads(l) for l in open('data/train_gt.jsonl')]:
        if r['audio'] in pseudo:
            f.write(json.dumps({'audio': r['audio'], 'text': pseudo[r['audio']], 'duration': r['duration']})+'\n')

from edge_asr.eval import wer
ev = [json.loads(l) for l in open('data/eval.jsonl')]
refs = [r['text'] for r in ev]; hyps = [pseudo.get(r['audio'],'') for r in ev]
print('Whisper (teacher) WER on eval:', round(wer(refs,hyps),3))


## 4. Train a BPE tokenizer on the transcripts
BPE-500 subwords — the standard text units for a transducer (cleaner than characters).


In [ ]:
from edge_asr.data.tokenizer import SentencePieceTokenizer
SentencePieceTokenizer.train('data/train_text.txt','data/bpe500', vocab_size=500)
print('tokenizer -> data/bpe500.model')


## 5. Train YOUR student on Whisper's labels (GPU)
Streaming transducer + aux CTC, ~11 M params. `--device auto` uses the GPU. 
Bump `--steps` for better accuracy; watch the loss fall.


In [ ]:
!PYTHONPATH=. python -m edge_asr.training.train_model1 \
    --config configs/model1_general.yaml --tokenizer data/bpe500.model \
    --manifest data/train_pseudo.jsonl --steps 5000 --batch-size 16 --lr 3e-4 \
    --device auto --out runs/student
# If you hit CUDA out-of-memory, lower --batch-size (e.g. 8) or MAX_SEC in cell 2.


## 6. Evaluate the student (vs ground truth)
Both full-context and streaming decode, so you can see the streaming penalty.


In [ ]:
!PYTHONPATH=. python -m edge_asr.eval.evaluate --ckpt runs/student/model1.pt --manifest data/eval.jsonl
!PYTHONPATH=. python -m edge_asr.eval.evaluate --ckpt runs/student/model1.pt --manifest data/eval.jsonl --streaming


### Qualitative — a few student transcriptions


In [ ]:
import json, torch
from edge_asr.data import ManifestDataset, load_tokenizer
from edge_asr.features import LogMelFrontend, OnlineCMVN
from edge_asr.decode import greedy_search
from edge_asr.training.utils import build_model1
blob = torch.load('runs/student/model1.pt', weights_only=False)
tok = load_tokenizer('data/bpe500.model')
m = build_model1(blob['config'], blob['vocab_size']); m.load_state_dict(blob['model']); m.eval()
fe = LogMelFrontend(n_mels=80); cmvn = OnlineCMVN(n_mels=80)
ds = ManifestDataset('data/eval.jsonl', fe, cmvn, tok, task='asr')
for i in range(8):
    feats, toks = ds[i]
    print('REF :', tok.decode(toks.tolist()))
    print('HYP :', tok.decode(greedy_search(m, feats)))
    print()


## 7. Export + int4-quantize → the shippable size
3-graph static ONNX (encoder / decoder / joiner) + int4-mixed. Prints the real on-disk MB vs the 10 MB budget.


In [ ]:
!PYTHONPATH=. python scripts/export_pipeline.py --ckpt runs/student/model1.pt \
    --out runs/student/onnx --mode mixed


## 8. (Optional) Save your model to Google Drive


In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# import shutil, os; os.makedirs('/content/drive/MyDrive/wristvoice', exist_ok=True)
# shutil.copytree('runs/student', '/content/drive/MyDrive/wristvoice/student', dirs_exist_ok=True)
# print('saved to Drive')


---
### What you just did
- Ran the **real** pipeline on real speech: **Whisper teacher → labels → your streaming student → int4 model**.
- The student file is a few MB — the shape/size that fits a watch.

### To get production accuracy (office GPU)
1. Use `train-clean-100` then `train-clean-360` (change `url=` in cell 2, raise `N_UTTS`).
2. Raise `--steps` (tens of thousands) and add your **wrist-domain recordings**.
3. Use a stronger teacher (`--model large-v3`) for labels.
4. Finish with **int4 QAT** (not just PTQ) before export.

Repo: https://github.com/codejawk/ASR
